# Across-night movement and HR response — one subject

All saved crops represent sleep period time (SPT), including wake within that period. No further sleep/wake selection is applied. The SPT denominator is the span of saved acceleration timestamps plus one sample period; exact diary boundaries are not saved in these exports. Acquisition gaps remain in SPT, are reported as reduced coverage, and are never bridged by filtering or HR windows.

AUC tertiles use common 1/3 and 2/3 quantiles across **all candidate bursts from all selected nights**, before HR exclusions. Ties stay together. Adding nights refits these thresholds. Small = Low; medium + large = Medium + High. Movement summaries include every detected burst; HR summaries/correlations include only eligible isolated events.

HR follows the existing notebook: **FIT-recorded HR**, baseline −20 to −6 s, full epoch −20 to +49 s, peak increase = max(0, maximum post-onset percentage change), latency to positive peak in +1 to +49 s. No positive peak means undefined latency. HR timing is approximate. Recovery is deliberately omitted.

Nights receive equal weight in across-night metric means and HR curves. Shading is between-night ±1 SD, describing variability in this subject, not population uncertainty. Correlations are descriptive; bursts cluster within nights and chronological nights may also be dependent. No independent-event p-values are reported.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

MODULE_DIR = next(candidate for base in [Path.cwd(), *Path.cwd().parents]
                  for candidate in [base, base / "offline_processing"]
                  if (candidate / "across_nights.py").exists())
sys.path.insert(0, str(MODULE_DIR))
from across_nights import analyze_subject

# Override with OVERNIGHT_PROCESSED_ROOT or edit this cell for another subject.
PROCESSED_ROOT = Path(os.environ.get("OVERNIGHT_PROCESSED_ROOT", str(Path.home() / "Documents/garmin/sleep/garmin_processed")))
NAIVE_TIMEZONE = "Europe/Rome"  # Saved exports contain naive local wall time.
OUTPUT_DIR = MODULE_DIR / "outputs" / "across_nights"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
COLORS = {"Low": "#167D8D", "Medium": "#A6761D", "High": "#B52E55"}


In [ ]:
result = analyze_subject(PROCESSED_ROOT, naive_timezone=NAIVE_TIMEZONE)
nights = result["nights"]
nightly = result["nightly"]
print(f"{len(nights)} nights; pooled AUC cutoffs: {result['cutoffs']} g·s")
display(nights)
display(result["events"].loc[lambda x: ~x.included].groupby(["night", "exclusion_reason"]).size().rename("excluded"))


## Nightly metrics and trends

Dates identify night folders by recording start. Points/lines connect nightly means in chronological order; the dashed line is the equal-night mean. Movement burden is the sum of detected burst durations divided by total SPT, including the gaps within merged bursts as defined by the existing detector. Missing group means stay missing, not zero. Counts, coverage, and HR eligibility should be checked before interpreting trends.

In [ ]:
metrics = {"p2p_mean_g": "Mean burst p2p (g)", "auc_mean_g_s": "Mean burst AUC (g·s)",
           "duration_mean_s": "Mean burst duration (s)", "burst_spt_pct": "Burst duration / SPT (%)",
           "hr_peak_mean_pct": "Mean HR peak increase (%)", "hr_latency_mean_s": "Mean peak latency (s)"}
summary = nightly.groupby("group", sort=False)[list(metrics)].agg(["mean", "std", "count"])
display(summary)
display(nightly)
fig, axes = plt.subplots(3, 2, figsize=(14, 12), constrained_layout=True)
for ax, (metric, label) in zip(axes.flat, metrics.items()):
    for group, color in {"All": "#333333", **COLORS}.items():
        data = nightly.loc[nightly.group == group].set_index("night").reindex(nights.index)
        dates = pd.to_datetime(data.index, format="%Y-%m-%d-%H-%M-%S")
        ax.plot(dates, data[metric], 'o-', color=color, label=group, alpha=.85)
    overall = nightly.loc[nightly.group == "All", metric].mean()
    ax.axhline(overall, color="black", linestyle="--", alpha=.4)
    ax.set(ylabel=label)
    ax.tick_params(axis="x", rotation=45)
axes.flat[0].legend(ncol=4)
plt.show()


## Full HR curves by common AUC tertile

Thin traces show each night's mean response. The thick curve averages those nightly means, giving each available night equal weight. Nights without eligible bursts in a tertile do not contribute. The table reports contributing nights and bursts.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True, constrained_layout=True)
for ax, (group, color) in zip(axes, COLORS.items()):
    local = result["night_curves"].query("group == @group")
    for night, curve in local.groupby("night"):
        ax.plot(curve.relative_s, curve.mean_pct, color=color, alpha=.16, linewidth=.8)
    grand = result["grand_curves"].query("group == @group")
    ax.plot(grand.relative_s, grand['mean'], color=color, linewidth=2.5)
    ax.fill_between(grand.relative_s, grand['mean'] - grand['std'], grand['mean'] + grand['std'], color=color, alpha=.18)
    ax.axvline(0, color="gray", linestyle="--")
    ax.axhline(0, color="gray", linewidth=.7)
    ax.axvspan(-20, -5, color="gray", alpha=.1)
    ax.set(title=f"{group} AUC — {local.night.nunique()} nights", xlabel="Seconds from burst onset")
axes[0].set_ylabel("HR change from baseline (%)")
plt.show()
display(nightly.pivot(index="night", columns="group", values="hr_n"))


## Movement–HR correlations: bursts and nightly averages

Both predictors use the same **AUC-defined** groups. Each burst plot pools eligible events (colour identifies night); each nightly point averages the movement metric and HR peak over the same eligible events within that size group. These answer different questions: event-level association versus covariation of nightly means. Pearson and Spearman coefficients are shown; fewer than three pairs or constant inputs yield missing coefficients. Pooled associations can reflect differences between nights.

In [ ]:
display(result["correlations"])
for level, table in [("Burst", result["burst_pairs"]), ("Night", result["night_pairs"])]:
    fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
    for row, group in enumerate(["Small", "Medium + large"]):
        subset = table.loc[table.size_group == group]
        for col, (metric, label) in enumerate([("AUC", "Movement AUC (g·s)"), ("peak-to-peak", "Movement p2p (g)")]):
            ax = axes[row, col]
            for i, night in enumerate(nights.index):
                points = subset.loc[subset.night == night]
                ax.scatter(points[metric], points.hr_peak_increase_pct, color=plt.cm.tab20(i % 20), alpha=.7, s=24)
            stats = result["correlations"].query("level == @level and group == @group and metric == @metric").iloc[0]
            ax.set(title=f"{level}: {group} | n={stats['n']} | r={stats.pearson_r:.2f}, ρ={stats.spearman_rho:.2f}",
                   xlabel=label, ylabel="HR peak increase (%)")
    plt.show()


## Local exports

Exports and executed notebook outputs contain personal data and remain in the Git-ignored outputs directory. Source notebooks are published with empty outputs. Re-running recomputes all nights and common thresholds.

In [ ]:
import json
for name in ["nights", "events", "nightly", "night_curves", "grand_curves", "night_pairs", "correlations"]:
    result[name].to_csv(OUTPUT_DIR / f"{name}.csv", index=name in ["nights", "events"])
(OUTPUT_DIR / "settings.json").write_text(json.dumps({
    "nights": list(nights.index), "common_auc_cutoffs_g_s": list(result["cutoffs"]),
    "naive_timezone": NAIVE_TIMEZONE, "sampling_rate_hz": 100, "threshold_g": .040,
    "max_accel_gap_s": .25, "merge_gap_s": 5, "hr_source": "FIT-recorded HR",
    "hr_limits_bpm": [30, 200], "max_hr_gap_s": 3, "isolation_s": 30,
    "exclude_late_overlap": True, "spt": "saved acceleration span plus sample period",
    "night_weighting": "equal", "recovery": False,
}, indent=2), encoding="utf-8")
print(f"Saved local tables to {OUTPUT_DIR}")
